# Logs

> Generating / formatting the base logging instance

In [ ]:
#| default_exp mcp

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
from __future__ import annotations
from typing import Any, Dict, List, Optional, Iterable, Tuple, Set

import os, sys, json
import argparse
from pathlib import Path

from mcp.server.fastmcp import FastMCP  # official MCP Python SDK (FastMCP 2.0+)

from nbdev_mcp import __version__
from nbdev_mcp.utils.logs import log
from nbdev_mcp.utils.config import CURRENT_PROJECT
from nbdev_mcp.utils.paths import resolve_selector
from nbdev_mcp.utils.subprocess import watch_notebooks
from nbdev_mcp.resources import add_resources
from nbdev_mcp.tools import (
    add_project_tools, add_env_tools, add_nbdev_tools, 
    add_notebook_editing_tools, add_lint_tools, 
    add_analysis_tools, add_tests_tools
)
from nbdev_mcp.prompts import add_prompts
from nbdev_mcp.tasks import add_task_tools, enable_nbdev_tasks

## HTTP Utils

In [ ]:
#| export
def set_http_path_if_supported(target_path: str) -> bool:
    """Try to set the HTTP mount path on the MCP SDK settings.
    
    Parameters
    ----------
    target_path : str
        The path to set (e.g., '/mcp').
    
    Returns
    -------
    bool
        True if successfully set, False if not supported.
    """
    import mcp
    try:
        mcp.settings.streamable_http_path = target_path  # type: ignore[attr-defined]
        return True
    except Exception:
        try:
            mcp.settings.http_path = target_path  # type: ignore[attr-defined]
            return True
        except Exception:
            return False

## Create the MCP

In [ ]:
#| export
def create_nbdev_mcp(name: str = "mcp.nbdev") -> FastMCP:
    """Create and configure the nbdev MCP server with all resources, tools, and prompts."""
    mcp = FastMCP(name)
    
    # Enable experimental tasks API
    enable_nbdev_tasks(mcp)
    
    # Attach all nbdev-related resources, tools, and prompts
    add_resources(mcp)
    add_project_tools(mcp)
    add_env_tools(mcp)
    add_nbdev_tools(mcp)
    add_notebook_editing_tools(mcp)  # CRITICAL: Notebook editing and workflow tools
    add_prompts(mcp)  # Philosophy prompts must come after tools are registered
    
    # Extensions: linting, analysis, and code generation tools
    add_lint_tools(mcp)
    add_analysis_tools(mcp)
    add_tests_tools(mcp)
    
    # Task-based tools for auditing, deduplication, and refactoring
    add_task_tools(mcp)
    
    return mcp

## `main`

In [ ]:
#| export
def main() -> None:
    """Entry point for the nbdev MCP server CLI."""
    parser = argparse.ArgumentParser(
        description="nbdev MCP server",
        formatter_class=argparse.RawTextHelpFormatter,
    )
    parser.add_argument("-V", "--version", action="version", version=f"%(prog)s {__version__}")
    
    subparsers = parser.add_subparsers(dest="command", help="Commands")
    
    # --- run subcommand (default behavior) ---
    run_parser = subparsers.add_parser("run", help="Run the MCP server (default)")
    run_parser.add_argument("--project", help="Path or alias for an nbdev project.")
    run_parser.add_argument("--transport", choices=("stdio", "http", "streamable-http"), 
                            default=os.environ.get("NBDEV_MCP_TRANSPORT", "stdio"),
                            help="Transport mode (default: stdio).")
    run_parser.add_argument("--host", default=os.environ.get("NBDEV_MCP_HOST", "127.0.0.1"))
    run_parser.add_argument("--port", type=int, default=int(os.environ.get("NBDEV_MCP_PORT", "8000")))
    run_parser.add_argument("--path", default=os.environ.get("NBDEV_MCP_PATH", "/mcp"))
    run_parser.add_argument("-v", "--verbose", action="store_true")
    run_parser.add_argument("-w", "--watch", action="store_true")
    run_parser.add_argument("--watch-interval", type=float, default=2.0)
    run_parser.add_argument("--watch-cmd", default="nbdev_export")
    
    # --- install subcommand ---
    install_parser = subparsers.add_parser("install", help="Install to Claude Desktop")
    install_parser.add_argument("--dry-run", action="store_true", help="Show what would be done")
    
    # --- uninstall subcommand ---
    uninstall_parser = subparsers.add_parser("uninstall", help="Remove from Claude Desktop")
    uninstall_parser.add_argument("--dry-run", action="store_true", help="Show what would be done")
    
    # For backwards compat: add run args to main parser too
    parser.add_argument("--project", help=argparse.SUPPRESS)
    parser.add_argument("--transport", choices=("stdio", "http", "streamable-http"), 
                        default=os.environ.get("NBDEV_MCP_TRANSPORT", "stdio"), help=argparse.SUPPRESS)
    parser.add_argument("--host", default=os.environ.get("NBDEV_MCP_HOST", "127.0.0.1"), help=argparse.SUPPRESS)
    parser.add_argument("--port", type=int, default=int(os.environ.get("NBDEV_MCP_PORT", "8000")), help=argparse.SUPPRESS)
    parser.add_argument("--path", default=os.environ.get("NBDEV_MCP_PATH", "/mcp"), help=argparse.SUPPRESS)
    parser.add_argument("-v", "--verbose", action="store_true", help=argparse.SUPPRESS)
    parser.add_argument("-w", "--watch", action="store_true", help=argparse.SUPPRESS)
    parser.add_argument("--watch-interval", type=float, default=2.0, help=argparse.SUPPRESS)
    parser.add_argument("--watch-cmd", default="nbdev_export", help=argparse.SUPPRESS)
    
    args = parser.parse_args()
    
    # Handle install/uninstall
    if args.command == "install":
        install_to_claude(dry_run=args.dry_run)
        return
    elif args.command == "uninstall":
        uninstall_from_claude(dry_run=args.dry_run)
        return
    
    # Default: run server (with or without 'run' subcommand)
    run_server(args)

## Next

In [ ]:
#| export


## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()

## Install helpers

In [ ]:
#| export
def get_claude_config_path() -> Path:
    """Get Claude Desktop config path for current platform."""
    if sys.platform == "darwin":
        return Path.home() / "Library/Application Support/Claude/claude_desktop_config.json"
    elif sys.platform == "win32":
        return Path(os.environ.get("APPDATA", "")) / "Claude/claude_desktop_config.json"
    else:  # Linux
        return Path.home() / ".config/Claude/claude_desktop_config.json"


def get_vscode_mcp_path() -> Path:
    """Get VS Code MCP settings path for current platform."""
    if sys.platform == "darwin":
        return Path.home() / "Library/Application Support/Code/User/settings.json"
    elif sys.platform == "win32":
        return Path(os.environ.get("APPDATA", "")) / "Code/User/settings.json"
    else:  # Linux
        return Path.home() / ".config/Code/User/settings.json"


def get_python_path() -> str:
    """Get current Python interpreter path."""
    return sys.executable


def install_to_claude(dry_run: bool = False) -> None:
    """Install nbdev-mcp to Claude Desktop configuration.
    
    Parameters
    ----------
    dry_run : bool
        If True, show what would be done without making changes.
    """
    config_path = get_claude_config_path()
    python_path = get_python_path()
    
    server_config = {
        "command": python_path,
        "args": ["-m", "nbdev_mcp"]
    }
    
    if dry_run:
        print(f"Would install to: {config_path}")
        print(f"Server config:")
        print(json.dumps({"mcpServers": {"nbdev": server_config}}, indent=2))
        return
    
    # Read existing config or create new
    if config_path.exists():
        config = json.loads(config_path.read_text())
    else:
        config_path.parent.mkdir(parents=True, exist_ok=True)
        config = {}
    
    # Add/update mcpServers
    if "mcpServers" not in config:
        config["mcpServers"] = {}
    config["mcpServers"]["nbdev"] = server_config
    
    # Write config
    config_path.write_text(json.dumps(config, indent=2))
    print(f"✓ Installed nbdev-mcp to {config_path}")
    print(f"  Python: {python_path}")
    print("  Restart Claude Desktop to activate.")


def uninstall_from_claude(dry_run: bool = False) -> None:
    """Remove nbdev-mcp from Claude Desktop configuration.
    
    Parameters
    ----------
    dry_run : bool
        If True, show what would be done without making changes.
    """
    config_path = get_claude_config_path()
    
    if not config_path.exists():
        print(f"Config not found: {config_path}")
        return
    
    config = json.loads(config_path.read_text())
    
    if "mcpServers" not in config or "nbdev" not in config["mcpServers"]:
        print("nbdev-mcp not installed in Claude config.")
        return
    
    if dry_run:
        print(f"Would remove 'nbdev' from mcpServers in: {config_path}")
        return
    
    del config["mcpServers"]["nbdev"]
    config_path.write_text(json.dumps(config, indent=2))
    print(f"✓ Removed nbdev-mcp from {config_path}")

In [ ]:
#| export
def run_server(args) -> None:
    """Run the MCP server with the given arguments."""
    # Configure logging
    if args.verbose:
        import logging
        logging.basicConfig(
            level=logging.DEBUG,
            format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
            datefmt="%Y-%m-%d %H:%M:%S"
        )
        log.setLevel(logging.DEBUG)
        log.debug(f"nbdev-mcp {__version__}, Python {sys.version}")

    # Set initial project if provided
    proj_path = None
    if args.project:
        try:
            proj_path = resolve_selector(args.project)
            if args.verbose:
                log.debug(f"Project: {proj_path}")
        except Exception as e:
            log.error(str(e))
            if args.watch:
                sys.exit(1)

    # Watch mode
    if args.watch:
        if not proj_path:
            log.error("Watch mode requires --project")
            sys.exit(1)
        watch_notebooks(proj_path, interval=args.watch_interval, on_change=args.watch_cmd)
        return

    # Build and run MCP server
    mcp = create_nbdev_mcp()
    
    default_host, default_port, default_path = "127.0.0.1", 8000, "/mcp"
    using_defaults = (args.host == default_host and args.port == default_port and args.path == default_path)

    match args.transport:
        case "stdio":
            mcp.run(transport="stdio")
        case "streamable-http":
            if using_defaults:
                mcp.run(transport="streamable-http")
            else:
                try:
                    import uvicorn
                except ImportError:
                    log.error("uvicorn required for custom HTTP. pip install uvicorn")
                    sys.exit(1)
                if args.path != default_path:
                    set_http_path_if_supported(args.path)
                app = mcp.streamable_http_app()
                uvicorn.run(app, host=args.host, port=args.port)
        case "http":
            try:
                import uvicorn
            except ImportError:
                log.error("uvicorn required for HTTP. pip install uvicorn")
                sys.exit(1)
            if args.path != default_path:
                set_http_path_if_supported(args.path)
            app = mcp.streamable_http_app()
            uvicorn.run(app, host=args.host, port=args.port)
        case _:
            raise SystemExit(f"Unknown transport: {args.transport!r}")